# Tensores de entrada e saida no RioNowcast

Este notebook explica como uma janela temporal de radar e observacoes de estacoes e convertida em tensores para nowcasting. Ele usa exemplos sinteticos por padrao e inclui uma inspecao opcional de uma amostra real.

## Objetivos

- Distinguir o formato de armazenamento do formato consumido pelo PyTorch.
- Entender a separacao temporal entre entrada, target e mascara.
- Ver como observacoes defasadas de estacoes podem ser adicionadas ao radar.
- Identificar os requisitos para novas fontes, como imagens do satelite GOES.

## Convencoes

Os memmaps de radar sao armazenados como `[tempo, altura, largura, canal]`. O dataset os reorganiza para `[canal, tempo, altura, largura]`; o DataLoader adiciona o eixo inicial de lote.

Para uma janela que comeca no frame `s`, com cinco passos de entrada e cinco de previsao:

```text
entrada: s, s+1, s+2, s+3, s+4
target:  s+5, s+6, s+7, s+8, s+9
```

Cada passo representa 15 minutos. Portanto, o modelo recebe 75 minutos de contexto e estima os 75 minutos seguintes.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import torch


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_ROOT = PROJECT_ROOT / 'src'
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

print('Projeto:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)

## Exemplo sintetico: radar e estacoes

O crop atual das estacoes AlertaRio mede `52 x 67` pixels. O radar possui tres canais RGB. Quando a opcao `--input-stations` e ativada, sao adicionados dois canais: chuva observada e mascara de disponibilidade.

In [ ]:
batch_size = 2
radar_channels = 3
station_channels = 2
t_in = 5
t_out = 5
height, width = 52, 67

# O radar ja chega normalizado para [0, 1] ao modelo.
radar = torch.rand(batch_size, radar_channels, t_in, height, width)

# Chuva das estacoes: zero fora dos pixels observados.
station_rain = torch.zeros(batch_size, 1, t_in, height, width)
station_mask = torch.zeros_like(station_rain)
station_rain[:, 0, 4, 12, 18] = np.log1p(4.0)
station_mask[:, 0, 4, 12, 18] = 1.0
station_rain[:, 0, 4, 31, 44] = np.log1p(15.0)
station_mask[:, 0, 4, 31, 44] = 1.0

# Fusao: radar RGB + valor de chuva + mascara de disponibilidade.
inputs_fusion = torch.cat((radar, station_rain, station_mask), dim=1)

# Targets futuros usam a mesma grade, mas outro intervalo temporal.
target = torch.zeros(batch_size, 1, t_out, height, width)
target_mask = torch.zeros_like(target)
target[:, 0, 0, 12, 18] = np.log1p(5.0)
target_mask[:, 0, 0, 12, 18] = 1.0

print('Radar apenas:       ', tuple(radar.shape))
print('Radar + estacoes:  ', tuple(inputs_fusion.shape))
print('Target futuro:     ', tuple(target.shape))
print('Mascara de target: ', tuple(target_mask.shape))

O formato geral usado pelo STConvS2S e:

```text
entrada: [lote, canais, tempo_entrada, altura, largura]
saida:   [lote, 1,      tempo_saida,   altura, largura]
```

No experimento C1, sem estacoes como entrada, a forma e `[B, 3, 5, 52, 67]`. Em uma execucao de fusao, ela passa a `[B, 5, 5, 52, 67]`. A saida nao muda: a precipitacao futura continua tendo um canal.

## Por que a mascara e necessaria?

Um zero no tensor de chuva pode significar `0 mm/15 min` em uma estacao ou ausencia de estacao naquele pixel. A mascara distingue os dois casos. A loss e as metricas consideram apenas as posicoes com mascara igual a um.

In [ ]:
prediction = torch.zeros_like(target)
absolute_error = (prediction - target).abs()
masked_mae = (absolute_error * target_mask).sum() / target_mask.sum()

print('Pixels no tensor de target:', target.numel())
print('Observacoes validas:', int(target_mask.sum()))
print('MAE mascarada no exemplo:', float(masked_mae))

# A unidade interna do target e log1p(mm/15 min).
print('Chuva observada no exemplo:', float(torch.expm1(target[target_mask.bool()][0])), 'mm/15 min')

## Inspecao opcional de uma amostra real

Defina `LOAD_REAL_SAMPLE = True` apenas em uma maquina que possua o dataset V2. Esta celula le uma unica amostra por memmap; ela nao carrega os frames completos na RAM.

In [ ]:
LOAD_REAL_SAMPLE = False
DEFAULT_DATASET_ROOT = PROJECT_ROOT / 'data' / 'datasets' / 'radar_sumare_2012_2024_15min_128_alertario_v2'
DATASET_ROOT = Path(os.environ.get('RIONOWCAST_DATASET_ROOT', DEFAULT_DATASET_ROOT))

if LOAD_REAL_SAMPLE:
    from nowcasting.dataset import RadarStationMemmapDataset

    dataset = RadarStationMemmapDataset(
        DATASET_ROOT, [2012], stride=5, target_source='alertario',
        crop_stations=True, crop_margin_pixels=20,
        input_stations=True,
        station_mapping=PROJECT_ROOT / 'data' / 'mapeamento_pixel_estacao_alertario.csv',
        split_name='tensor-demo',
    )
    x_real, y_real, m_real = dataset[0]
    print('X real:', tuple(x_real.shape), x_real.dtype)
    print('Y real:', tuple(y_real.shape), y_real.dtype)
    print('M real:', tuple(m_real.shape), m_real.dtype)
    print('Observacoes futuras na primeira janela:', int(m_real.sum()))
else:
    print('Inspecao real desativada. Defina LOAD_REAL_SAMPLE = True para habilita-la.')

## Extensao para novas fontes

O STConvS2S aceita qualquer numero de canais. Uma nova fonte deve produzir canais alinhados a mesma janela temporal e grade espacial do radar. Cada variavel com ausencia espacial ou temporal deve, em geral, vir acompanhada de uma mascara.

| Fonte | Canais exemplares | Requisitos antes da fusao |
|---|---:|---|
| Radar Sumaré | 3 RGB | agregacao de 15 min e normalizacao para `[0, 1]` |
| AlertaRio defasado | 2 | `log1p(m15)` e mascara, sem usar instantes futuros |
| GOES | 1 ou mais por banda | reprojecao para a grade do radar, alinhamento temporal e normalizacao por banda |
| Produtos derivados | 1 ou mais | documentar origem, janela temporal, escala e tratamento de ausencias |

Uma configuracao futura poderia formar a entrada abaixo:

```text
[R, G, B, chuva_estacoes, mascara_estacoes, GOES_IR, GOES_WV, mascara_GOES]
```

Para cada nova fonte, valide: (1) mesma grade ou reprojecao documentada; (2) timestamp anterior ao horizonte previsto; (3) normalizacao ajustada apenas com dados de treino; (4) mascara para ausencias; e (5) experimento de ablacao que isole sua contribuicao.

## Exercicios sugeridos

1. Altere `t_in` e `t_out` no exemplo sintetico e acompanhe as formas resultantes.
2. Remova a mascara de estacoes e discuta por que zero deixa de ser interpretavel.
3. Adicione um canal sintetico de GOES e verifique que apenas o eixo de canais muda.
4. Compare os experimentos radar-only, estacoes-only e radar mais estacoes mantendo o mesmo recorte temporal e os mesmos targets V2.